<a href="https://colab.research.google.com/github/atharvsalokhe30/Deep_Learning-_project_practice/blob/main/LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LogisticRegressionDemo") \
    .getOrCreate()

In [ ]:
data = [
    (0, 1.0, 2.0),
    (1, 2.0, 1.0),
    (0, 1.5, 1.3),
    (1, 3.0, 2.5),
    (0, 0.5, 0.7)
]

columns = ["label", "feature1", "feature2"]
df = spark.createDataFrame(data, columns)
df.show()

+-----+--------+--------+
|label|feature1|feature2|
+-----+--------+--------+
|    0|     1.0|     2.0|
|    1|     2.0|     1.0|
|    0|     1.5|     1.3|
|    1|     3.0|     2.5|
|    0|     0.5|     0.7|
+-----+--------+--------+



In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["feature1", "feature2"],
    outputCol="features"
)

df = assembler.transform(df)
df.select("features", "label").show()

+---------+-----+
| features|label|
+---------+-----+
|[1.0,2.0]|    0|
|[2.0,1.0]|    1|
|[1.5,1.3]|    0|
|[3.0,2.5]|    1|
|[0.5,0.7]|    0|
+---------+-----+



In [ ]:
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_data)

In [ ]:
predictions = model.transform(test_data)
predictions.select("features", "label", "prediction", "probability").show()

+---------+-----+----------+-----------+
| features|label|prediction|probability|
+---------+-----+----------+-----------+
|[0.5,0.7]|    0|       0.0|  [1.0,0.0]|
+---------+-----+----------+-----------+



In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)

print("AUC:", auc)


AUC: 0.0


In [ ]:
print("Coefficients:", model.coefficients)
print("Intercept:", model.intercept)

Coefficients: [50.78275422359652,-31.304365317633504]
Intercept: -52.687109199834055
